# 01 - Integración y limpieza de fuentes
**Proyecto:** StreamView Analytics - EP1 ADY1104 Visualización de Datos (Duoc UC)  
**Equipo:** Claudio Aro, Guillermo Cerda, Manuel Díaz

Objetivo: describir las dos fuentes de datos, evaluar su calidad e integrarlas en un catálogo único listo para visualizar.

In [1]:
import sys
from pathlib import Path
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))
import pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)

## 1. Carga de las fuentes originales

In [2]:
from src import preparacion as prep
peliculas, series = prep.cargar_fuentes()
print("Películas:", peliculas.shape, "| Series:", series.shape)
peliculas.drop(columns=["title", "director", "cast", "description"]).head(3)

Películas: (16000, 18) | Series: (16000, 16)


,show_id,type,country,date_added,release_year,rating,duration,genres,language,popularity,vote_count,vote_average,budget,revenue
0,10192,Movie,United States of America,2010-05-16,2010,6.380,NaN,"Comedy, Adventure, Fantasy, Animation, Family",en,203.893,7449,6.380,165000000,752600867
1,27205,Movie,"United Kingdom, United States of America",2010-07-15,2010,8.369,NaN,"Action, Science Fiction, Adventure",en,156.242,37119,8.369,160000000,839030630
2,12444,Movie,"United Kingdom, United States of America",2010-11-17,2010,7.744,NaN,"Adventure, Fantasy",en,121.191,19327,7.744,250000000,954305868


In [3]:
series.drop(columns=["title", "director", "cast", "description"]).head(3)

,show_id,type,country,date_added,release_year,rating,duration,genres,language,popularity,vote_count,vote_average
0,33238,TV Show,South Korea,2010-07-11,2010,8.241,1 Seasons,"Comedy, Reality",ko,1929.898,187,8.241
1,32415,TV Show,United States of America,2010-11-08,2010,7.035,1 Seasons,"Talk, Comedy, News",en,1670.580,229,7.035
2,37757,TV Show,Greece,2010-10-03,2010,5.600,1 Seasons,Reality,el,1317.092,6,5.600


## 2. Diagnóstico de calidad de datos

In [4]:
calidad = pd.DataFrame({
    "nulos_peliculas": peliculas.isna().sum(),
    "nulos_series": series.isna().sum().reindex(peliculas.columns),
})
calidad

,nulos_peliculas,nulos_series
show_id,0,0.0
type,0,0.0
title,0,0.0
director,132,10965.0
cast,204,1157.0
country,466,1797.0
date_added,0,0.0
release_year,0,0.0
rating,0,0.0
duration,16000,0.0


In [5]:
print("rating == vote_average (películas):", (peliculas.rating == peliculas.vote_average).mean())
print("rating == vote_average (series):   ", (series.rating == series.vote_average).mean())
print("duration en películas (no nulos):", peliculas.duration.notna().sum())
print("duration en series (valores únicos):", series.duration.unique())
print("Año de date_added == release_year:",
      (pd.to_datetime(peliculas.date_added).dt.year == peliculas.release_year).mean())
print("Títulos por año (películas):", peliculas.release_year.value_counts().unique())
print("Series con 0 votos:", (series.vote_count == 0).sum())
print("Películas con presupuesto informado:", (peliculas.budget > 0).mean().round(3))
print("ids duplicados en series:", series.show_id.duplicated().sum())

rating == vote_average (películas): 1.0
rating == vote_average (series):    1.0
duration en películas (no nulos): 0
duration en series (valores únicos): <ArrowStringArray>
['1 Seasons']
Length: 1, dtype: str
Año de date_added == release_year: 1.0
Títulos por año (películas): [1000]
Series con 0 votos: 3674
Películas con presupuesto informado: 0.303
ids duplicados en series: 9


**Decisiones de limpieza (resumen):**

| Problema detectado | Decisión |
|---|---|
| `rating` idéntico a `vote_average` | Se elimina `rating` (redundante) |
| `duration` vacía (películas) y constante (series) | Se excluye: no aporta información |
| Año de `date_added` = `release_year` en 100% | Se usa solo `release_year` |
| Calificación 0 con 0 votos | Se trata como "sin evaluar" (nulo), no como nota 0 |
| Presupuesto/recaudación en 0 | Nulo (no informado); ROI solo con montos >= USD 100.000 |
| 9 ids duplicados en series | Se eliminan |
| Taxonomías de género distintas | Se unifican en 21 géneros en español (+ "Sin clasificar"; 16 por tipo) |
| Textos con escrituras no latinas | Se normalizan al teclado latinoamericano |
| Muestra de 1.000 títulos por año y tipo | No se analiza crecimiento del volumen del catálogo |

## 3. Integración

In [6]:
cat, generos, paises = prep.ejecutar()
cat.head()

Catálogo integrado: 31,991 títulos (duplicados eliminados: 9)
Relaciones título-género: 65,788
Relaciones título-país:   37,628


,id,tipo,titulo,titulo_latino,anio,generos,genero_principal,idioma_cod,idioma,paises,pais_principal,pais_principal_en,director,reparto,popularidad,votos,visibilidad,calificacion,calif_ponderada,presupuesto,recaudacion,roi,descripcion
0,10192,Película,Shrek Forever After,True,2010,"Comedia, Acción y aventura, Ciencia ficción y ...",Comedia,en,Inglés,United States of America,Estados Unidos,United States of America,Mike Mitchell,"Mike Myers, Eddie Murphy, Cameron Diaz, Antoni...",203.893,7449,Muy alta (1000+),6.380,6.379,165000000.0,752600867.0,3.561217,A bored and domesticated Shrek pacts with deal...
1,27205,Película,Inception,True,2010,"Acción y aventura, Ciencia ficción y fantasía",Acción y aventura,en,Inglés,"United Kingdom, United States of America",Reino Unido,United Kingdom,Christopher Nolan,"Leonardo DiCaprio, Joseph Gordon-Levitt, Ken W...",156.242,37119,Muy alta (1000+),8.369,8.360,160000000.0,839030630.0,4.243941,"Cobb, a skilled thief who commits corporate es..."
2,12444,Película,Harry Potter and the Deathly Hallows: Part 1,True,2010,"Acción y aventura, Ciencia ficción y fantasía",Acción y aventura,en,Inglés,"United Kingdom, United States of America",Reino Unido,United Kingdom,David Yates,"Daniel Radcliffe, Emma Watson, Rupert Grint, T...",121.191,19327,Muy alta (1000+),7.744,7.733,250000000.0,954305868.0,2.817223,"Harry, Ron and Hermione walk away from their l..."
3,38757,Película,Tangled,True,2010,"Animación, Familiar, Acción y aventura",Animación,en,Inglés,United States of America,Estados Unidos,United States of America,"Byron Howard, Nathan Greno","Mandy Moore, Zachary Levi, Donna Murphy, Ron P...",111.762,11638,Muy alta (1000+),7.600,7.583,260000000.0,592461732.0,1.278699,"Feisty teenager Rapunzel, who has long and mag..."
4,10191,Película,How to Train Your Dragon,True,2010,"Ciencia ficción y fantasía, Acción y aventura,...",Ciencia ficción y fantasía,en,Inglés,United States of America,Estados Unidos,United States of America,"Chris Sanders, Dean DeBlois","Jay Baruchel, Gerard Butler, Craig Ferguson, A...",110.044,13259,Muy alta (1000+),7.800,7.783,165000000.0,494879471.0,1.999270,As the son of a Viking leader on the cusp of m...


In [7]:
cat.info()

<class 'pandas.DataFrame'>
RangeIndex: 31991 entries, 0 to 31990
Data columns (total 23 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 31991 non-null  int64  
 1   tipo               31991 non-null  str    
 2   titulo             31991 non-null  str    
 3   titulo_latino      31991 non-null  bool   
 4   anio               31991 non-null  int64  
 5   generos            31991 non-null  str    
 6   genero_principal   31991 non-null  object 
 7   idioma_cod         31991 non-null  str    
 8   idioma             31991 non-null  str    
 9   paises             29730 non-null  str    
 10  pais_principal     31991 non-null  str    
 11  pais_principal_en  29730 non-null  object 
 12  director           31991 non-null  str    
 13  reparto            31991 non-null  str    
 14  popularidad        31991 non-null  float64
 15  votos              31991 non-null  int64  
 16  visibilidad        31991 non-null

## 4. Variables derivadas clave
- **calif_ponderada**: (v/(v+m))*R + (m/(v+m))*C, con m = mediana de votos y C = media del tipo. Evita que un título con 2 votos y nota 10 supere a uno con 20.000 votos y nota 8,5.
- **visibilidad**: tramo de votos recibidos (Sin votos, Baja, Media, Alta, Muy alta).
- **roi**: (recaudación - presupuesto) / presupuesto, solo películas.

In [8]:
cat.groupby("tipo")[["votos", "calificacion", "calif_ponderada", "popularidad"]].describe().T.round(2)

tipo                   Película     Serie
votos           count  16000.00  15991.00
                mean     718.66    107.07
                std     2080.20    607.63
                min        0.00      0.00
                25%       53.00      1.00
                50%      138.00      4.00
                75%      422.00     30.00
                max    37119.00  24664.00
calificacion    count  15106.00  12325.00
                mean       6.31      7.02
                std        1.02      1.59
                min        0.00      0.00
                25%        5.73      6.30
                50%        6.40      7.20
                75%        7.00      8.00
                max       10.00     10.00
calif_ponderada count  15106.00  12325.00
                mean       6.38      7.12
                std        0.49      0.59
                min        3.46      3.60
                25%        6.11      6.77
                50%        6.32      7.12
                75%        6.62      7.47
                max        8.49      9.03
popularidad     count  16000.00  15991.00
                mean      20.38     64.88
                std       68.61    139.42
                min        3.86      2.32
                25%        7.84     24.90
                50%       10.91     36.20
                75%       17.34     62.22
                max     3876.01   6421.92

In [9]:
generos.groupby(["tipo", "genero"]).size().unstack(0).fillna(0).astype(int).sort_values("Película", ascending=False)

tipo,Película,Serie
genero,,
Drama,6910,7859
Comedia,4533,4572
Acción y aventura,4157,1988
Suspenso,3769,0
Ciencia ficción y fantasía,2730,1958
Romance,2571,0
Terror,2425,0
Crimen,1738,1465
Animación,1579,2490


## 5. Verificación de la taxonomía unificada
Conteo de géneros distintos tras la armonización (valida la cifra declarada en el informe).

In [10]:
g_pel = set(generos.loc[generos.tipo == "Película", "genero"]) - {"Sin clasificar"}
g_ser = set(generos.loc[generos.tipo == "Serie", "genero"]) - {"Sin clasificar"}
print("Géneros en películas:", len(g_pel), "| en series:", len(g_ser), "| total unificado:", len(g_pel | g_ser))
print("Títulos con escritura no latina:", (~cat.titulo_latino).sum())

Géneros en películas: 16 | en series: 16 | total unificado: 21
Títulos con escritura no latina: 704
